# Meta Model - Ensembling Five Models using Stacking


## Imports

In [22]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report


In [23]:
RANDOM_STATE = 42
target_col = 'Depression'

## Load Base Models

In [24]:
dt_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/dunith_decision_tree/decision_tree_model.joblib")
rf_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211009_Themiya_random_forrest/random_forest_student_depression.joblib")
svm_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211011_kaveesha_svm_model/best_svm_model.joblib")
gb_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/gbc_model.joblib")
lr_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/Rushani_Logistic_Regression/final_logistic_model.joblib")

models = {
    'DecisionTree': dt_model,
    'RandomForest': rf_model,
    'SVM': svm_model,
    'GradientBoosting': gb_model,
    'LogisticRegression': lr_model
}

## Load Dataset 

In [25]:
df = pd.read_csv("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/preprocessed_student_depression(GBC-model).csv")  
X = df.drop(columns=[target_col])
y = df[target_col]

## Split Dataset

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## Feature Engineering 

### Identify feature types

In [27]:
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

cols_to_scale = [col for col in numerical_cols if col != target_col]


### Scale Numerical Features

In [28]:
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_scaled[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test_scaled[cols_to_scale])


### One-Hot Encode Categorical Features

In [29]:
encoder = OneHotEncoder(handle_unknown='ignore', drop='first')

X_train_cat = encoder.fit_transform(X_train_scaled[categorical_cols])
X_test_cat = encoder.transform(X_test_scaled[categorical_cols])

# Convert to DataFrames
encoded_col_names = encoder.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat.toarray(), columns=encoded_col_names, index=X_train_scaled.index)
X_test_cat_df = pd.DataFrame(X_test_cat.toarray(), columns=encoded_col_names, index=X_test_scaled.index)


## Combining

In [30]:
X_train_final = pd.concat([X_train_scaled[cols_to_scale], X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test_scaled[cols_to_scale], X_test_cat_df], axis=1)

print("Train shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)

Train shape: (22293, 46)
Test shape: (5574, 46)


## Generate Meta Features

In [31]:
meta_train = pd.DataFrame()
meta_test = pd.DataFrame()

for name, model in models.items():
    print(f"Generating predictions from {name}...")
    try:
        meta_train[name] = model.predict_proba(X_train_final)[:, 1]
        meta_test[name] = model.predict_proba(X_test_final)[:, 1]
    except Exception as e:
        print(f"[X] {name} failed predict_proba, using predict() instead: {e}")
        meta_train[name] = model.predict(X_train_final)
        meta_test[name] = model.predict(X_test_final)


Generating predictions from DecisionTree...
Generating predictions from RandomForest...
Generating predictions from SVM...


/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/opt/conda/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(


Generating predictions from GradientBoosting...
Generating predictions from LogisticRegression...


In [32]:
print(meta_train.head())

   DecisionTree  RandomForest       SVM  GradientBoosting  LogisticRegression
0      0.845659      0.929186  0.972918          0.952250            0.954720
1      0.990446      0.915622  0.959188          0.937691            0.954335
2      0.163399      0.207950  0.341262          0.189518            0.193628
3      0.108696      0.400157  0.352386          0.109649            0.077332
4      0.847826      0.811042  0.907999          0.771058            0.850078


## Train Meta Model

In [33]:
meta_model = LogisticRegression(C=0.1,penalty='l2',solver='lbfgs',max_iter=10000, random_state=42)
meta_model.fit(meta_train, y_train)

y_pred_stack = meta_model.predict(meta_test)
y_proba_stack = meta_model.predict_proba(meta_test)[:, 1]

In [34]:
print("\n=== STACKING META MODEL PERFORMANCE ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_stack):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_stack):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_stack):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_stack):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_proba_stack):.4f}")

print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred_stack))



=== STACKING META MODEL PERFORMANCE ===
Accuracy:  0.9428
Precision: 0.9444
Recall:    0.9586
F1 Score:  0.9515
ROC AUC:   0.9841

Detailed Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.92      0.93      2312
           1       0.94      0.96      0.95      3262

    accuracy                           0.94      5574
   macro avg       0.94      0.94      0.94      5574
weighted avg       0.94      0.94      0.94      5574



In [38]:
overlap = set(meta_train.index).intersection(set(meta_test.index))
print(len(overlap))


5574


In [20]:
from sklearn.model_selection import KFold
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

meta_train = np.zeros((len(X_train_final), len(models)))
meta_test = np.zeros((len(X_test_final), len(models)))

for i, (name, model) in enumerate(models.items()):
    print(f"OOF for {name}...")
    test_fold_preds = np.zeros((len(X_test_final), kf.n_splits))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_final)):
        X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        meta_train[val_idx, i] = model.predict_proba(X_val)[:, 1]
        test_fold_preds[:, fold] = model.predict_proba(X_test_final)[:, 1]
    
    meta_test[:, i] = test_fold_preds.mean(axis=1)

# Convert to DataFrame
meta_train_df = pd.DataFrame(meta_train, columns=models.keys())
meta_test_df = pd.DataFrame(meta_test, columns=models.keys())


OOF for DecisionTree...
OOF for RandomForest...
OOF for SVM...
OOF for GradientBoosting...
OOF for LogisticRegression...


In [36]:
meta_model.fit(meta_train_df, y_train)
y_pred = meta_model.predict(meta_test_df)
y_proba = meta_model.predict_proba(meta_test_df)[:, 1]

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.8444564047362756
F1 Score: 0.868417058734254
ROC AUC: 0.9217350787828387

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.80      0.81      2312
           1       0.86      0.88      0.87      3262

    accuracy                           0.84      5574
   macro avg       0.84      0.84      0.84      5574
weighted avg       0.84      0.84      0.84      5574



In [39]:
for name, base in models.items():
    base.fit(X_train_final, y_train)
    meta_test_df[f"{name}_proba"] = base.predict_proba(X_test_final)[:,1]


In [40]:
from sklearn.model_selection import GridSearchCV
param_grid = {"C":[0.05,0.1,0.5,1],"solver":["lbfgs","liblinear"]}
grid = GridSearchCV(LogisticRegression(max_iter=10000),param_grid,cv=5,scoring="f1")
grid.fit(meta_train_df, y_train)
meta_model = grid.best_estimator_

In [45]:
from sklearn.metrics import precision_recall_curve, f1_score
proba = meta_model.predict_proba(meta_train_df)[:,1]
prec, rec, thr = precision_recall_curve(y_train, proba)
f1 = 2*prec*rec/(prec+rec+1e-12)
best_t = thr[f1[:-1].argmax()]

In [47]:
# 1) Get the expected columns & order from the trained meta-model
expected = list(meta_model.feature_names_in_)

# 2) Reindex test meta-features to the exact same columns/order
meta_test_fix = meta_test_df.reindex(columns=expected, fill_value=0)

# 3) Predict safely
y_pred_proba = meta_model.predict_proba(meta_test_fix)[:, 1]

y_pred_default = (y_pred_proba >= 0.5).astype(int)   # default 0.5 threshold

In [48]:
acc = accuracy_score(y_test, y_pred_default)
prec = precision_score(y_test, y_pred_default)
rec = recall_score(y_test, y_pred_default)
f1 = f1_score(y_test, y_pred_default)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=== STACKING META MODEL PERFORMANCE ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}\n")

print("Detailed Classification Report:")
print(classification_report(y_test, y_pred_default))

=== STACKING META MODEL PERFORMANCE ===
Accuracy:  0.8452
Precision: 0.8605
Recall:    0.8777
F1 Score:  0.8690
ROC AUC:   0.9218

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.80      0.81      2312
           1       0.86      0.88      0.87      3262

    accuracy                           0.85      5574
   macro avg       0.84      0.84      0.84      5574
weighted avg       0.84      0.85      0.84      5574



## Improve Model Accuracy

###  Add Meta Features

In [ ]:
import numpy as np

def add_meta_features(df):
    for col in df.columns:
        df[col + "_abs"] = np.abs(df[col] - 0.5)

    df["entropy"] = -(
        df * np.log(df + 1e-9) + (1 - df) * np.log(1 - df + 1e-9)
    ).mean(axis=1)

    df["mean_proba"] = df.mean(axis=1)
    df["std_proba"] = df.std(axis=1)

    return df

meta_train_enh = add_meta_features(meta_train.copy())
meta_test_enh = add_meta_features(meta_test.copy())


### Hyperparameter Tuning

In [52]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    'C': [0.0005, 0.001, 0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [5000]
}

lr = LogisticRegression()

grid = GridSearchCV(lr, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(meta_train_enh, y_train)

meta_model = grid.best_estimator_
print("Best meta-model:", grid.best_params_)


Best meta-model: {'C': 10, 'max_iter': 5000, 'penalty': 'l2', 'solver': 'liblinear'}


In [56]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

y_pred = meta_model.predict(meta_test_enh)
y_proba = meta_model.predict_proba(meta_test_enh)[:, 1]

print("=== STACKING FINAL META MODEL PERFORMANCE ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n", classification_report(y_test, y_pred))


=== STACKING FINAL META MODEL PERFORMANCE ===
Accuracy: 0.9590958019375673
Precision: 0.9608140947752126
Recall: 0.969650521152667
F1 Score: 0.965212084223375
ROC AUC: 0.9867603567556789

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.94      0.95      2312
           1       0.96      0.97      0.97      3262

    accuracy                           0.96      5574
   macro avg       0.96      0.96      0.96      5574
weighted avg       0.96      0.96      0.96      5574



### Save Model

In [57]:
joblib.dump(meta_model, "final_meta_model.joblib")
meta_train.to_csv("meta_train_predictions.csv", index=False)
meta_test.to_csv("meta_test_predictions.csv", index=False)